# Phase 3 - ML Experiment Matrix + Model Risk Management

**Intended use:** Train and compare models across splits and horizons, select a recall-first champion, document fairness and explanations.

**Prerequisite:** Phase 2 (`model_features.parquet`, `rnn_sequences.parquet`, feature dictionary).

**Primary protocol:** 70/15/15 split, 30-day horizon. Uses full gold lake by default (`MATRIX_SAMPLE=0`, `CHAMPION_SAMPLE=0`). Set env vars to a positive integer (e.g. 20000) for faster dev runs.


## 1. Setup

Registry helpers and project root.


In [1]:
from __future__ import annotations
import json, hashlib, os, re, uuid, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    ROOT = Path("..").resolve()
os.chdir(ROOT)
DATAFILE = ROOT / "datafile.txt"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def load_registry(path=DATAFILE):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 4:
            continue
        rows.append({"role": parts[0].strip(), "zone": parts[1].strip(), "path": parts[2].strip(), "description": parts[3].strip()})
    return pd.DataFrame(rows)

def registry_paths(role=None):
    reg = load_registry()
    if role is not None:
        reg = reg[reg["role"] == role]
    return [ROOT / p for p in reg["path"].tolist()]

def registry_path(role, default=None):
    paths = registry_paths(role=role)
    if paths:
        return paths[0]
    return ROOT / default if default else None

def upsert_registry(role, zone, rel_path, description):
    lines = DATAFILE.read_text(encoding="utf-8").splitlines()
    new_line = f"{role}|{zone}|{rel_path}|{description}"
    out, found = [], False
    for line in lines:
        if line.startswith("#") or not line.strip():
            out.append(line)
            continue
        parts = line.split("|")
        if len(parts) >= 3 and parts[0].strip() == role and parts[2].strip() == rel_path:
            out.append(new_line)
            found = True
        else:
            out.append(line)
    if not found:
        out.append(new_line)
    DATAFILE.write_text("\n".join(out) + "\n", encoding="utf-8")

def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_checksum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("ROOT:", ROOT)
print(load_registry())


ROOT: E:\Amit\Project\Hospital project
           role    zone                                       path  \
0           raw  bronze                 data/raw/diabetic_data.csv   
1         clean  silver        data/lake/silver/encounters.parquet   
2      features    gold      data/lake/gold/model_features.parquet   
3     sequences    gold       data/lake/gold/rnn_sequences.parquet   
4   predictions    gold    data/lake/gold/test_predictions.parquet   
5       metrics    gold  data/lake/gold/experiment_results.parquet   
6          mart  export          data/exports/mart_readmission.csv   
7          mart  export        data/exports/mart_clinical_risk.csv   
8          mart  export    data/exports/mart_model_performance.csv   
9          mart  export         data/exports/mart_dq_scorecard.csv   
10     manifest     ops    data/lake/bronze/_manifests/latest.json   
11     champion     ops              models/champion_register.json   
12  hyperparams     ops                    models/h

## 2. ML imports

Scikit-learn, boosting libraries, optional torch for RNN.


In [2]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, brier_score_loss)
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


## 3. Load gold data and experiment config

Features, sequences, split definitions, horizons (30d primary; 60/90 proxy labels).


In [3]:
feat_path = registry_path("features", "data/lake/gold/model_features.parquet")
seq_path = registry_path("sequences", "data/lake/gold/rnn_sequences.parquet")
df = pd.read_parquet(feat_path)
seq_df = pd.read_parquet(seq_path)
print("features", df.shape)

# Full lake by default (MATRIX_SAMPLE=0 / CHAMPION_SAMPLE=0). Set env to positive int for dev subsample only.
from ml.sample import sample_rows
from ml.tuning import make_estimator, load_hyperparams, get_rnn_params, HYPERPARAMS_PATH

MATRIX_SAMPLE = int(os.environ.get("MATRIX_SAMPLE", "0"))
CHAMPION_SAMPLE = int(os.environ.get("CHAMPION_SAMPLE", "0"))
PRIMARY_SPLIT = "70/15/15"
PRIMARY_HORIZON = "30d"
SPLITS = {
    "80/20": (0.80, None),
    "70/30": (0.70, None),
    "75/25": (0.75, None),
    "65/35": (0.65, None),
    "60/40": (0.60, None),
    "60/20/20": (0.60, 0.20),
    "70/15/15": (0.70, 0.15),
}
HORIZONS = {
    "30d": "readmit_30d",
    "60d": "readmit_60d_proxy",
    "90d": "readmit_90d_proxy",
}

META_COLS = ["encounter_id", "patient_nbr", "readmit_30d", "readmit_60d_proxy", "readmit_90d_proxy"]
# fairness cols may also be features
feature_dictionary = json.loads((ROOT / "data" / "nosql" / "feature_dictionary.json").read_text(encoding="utf-8"))
FEATURE_COLS = [c for c in feature_dictionary["features"] if c in df.columns]

features (101766, 30)


## 4. Helper functions

Train/val/test splits, metrics, recall-first threshold tuning, risk bands, model factories, optional RNN.


In [4]:
def make_split(data, y, split_name):
    train_frac, val_frac = SPLITS[split_name]
    if val_frac is None:
        X_train, X_test, y_train, y_test = train_test_split(
            data, y, train_size=train_frac, stratify=y, random_state=RANDOM_STATE)
        return X_train, None, X_test, y_train, None, y_test
    # three-way
    X_train, X_temp, y_train, y_temp = train_test_split(
        data, y, train_size=train_frac, stratify=y, random_state=RANDOM_STATE)
    relative_val = val_frac / (1 - train_frac)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, train_size=relative_val, stratify=y_temp, random_state=RANDOM_STATE)
    return X_train, X_val, X_test, y_train, y_val, y_test

def build_preprocessor(X):
    cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num_cols = [c for c in X.columns if c not in cat_cols]
    transformers = []
    if num_cols:
        transformers.append(("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num_cols))
    if cat_cols:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=30)),
        ]), cat_cols))
    pre = ColumnTransformer(transformers)
    return pre, cat_cols, num_cols

def metrics_dict(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan"),
    }

def best_threshold(y_true, y_prob, min_precision=0.15):
    # recall-first: maximize recall with mild precision floor
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t, best_rec, best_prec = 0.5, -1, 0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        if rec > best_rec and prec >= min_precision:
            best_t, best_rec, best_prec = float(t), rec, prec
    if best_rec < 0:
        best_t = 0.3
    return best_t

def risk_band(p):
    if p < 0.33:
        return "Low"
    if p < 0.66:
        return "Medium"
    return "High"

# Optional torch RNN
HAS_TORCH = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception:
    HAS_TORCH = False
    print("torch unavailable; RNN will be skipped")

class SmallRNN(nn.Module if HAS_TORCH else object):
    def __init__(self, vocab, emb=16, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.rnn = nn.GRU(emb, hidden, batch_first=True)
        self.fc = nn.Linear(hidden + 3, 1)
    def forward(self, seq, static):
        x = self.emb(seq)
        _, h = self.rnn(x)
        h = h.squeeze(0)
        z = torch.cat([h, static], dim=1)
        return self.fc(z).squeeze(1)

def train_rnn(seq_train, y_train, seq_val, y_val, token_maps, emb=16, hidden=32, lr=0.001, epochs=3):
    if not HAS_TORCH:
        return None
    device = torch.device("cpu")
    model = SmallRNN(len(token_maps["tok2id"]), emb=emb, hidden=hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    seq_t = torch.tensor(np.stack(seq_train["seq_ids"].to_numpy()), dtype=torch.long)
    st_t = torch.tensor(seq_train[["static_los", "static_visits", "static_meds"]].to_numpy(np.float32))
    y_t = torch.tensor(y_train.to_numpy(np.float32))
    loader = DataLoader(TensorDataset(seq_t, st_t, y_t), batch_size=256, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, sb, yb in loader:
            opt.zero_grad()
            logit = model(xb, sb)
            loss = loss_fn(logit, yb)
            loss.backward()
            opt.step()
    model.eval()
    return model

def build_seq_frame(base_df, seq_raw):
    # map tokens
    tokens = []
    for cols in [["diag_1","diag_2","diag_3"], ["insulin","metformin"]]:
        pass
    all_toks = set(["<pad>", "<unk>"])
    for c in ["diag_1","diag_2","diag_3","insulin","metformin"]:
        all_toks.update(seq_raw[c].fillna("<unk>").astype(str).unique().tolist())
    tok2id = {t:i for i,t in enumerate(sorted(all_toks))}
    tok2id["<pad>"] = 0
    def row_ids(r):
        seq = [tok2id.get(str(r[c]), tok2id["<unk>"]) for c in ["diag_1","diag_2","diag_3","insulin","metformin"]]
        return seq
    out = seq_raw.copy()
    out["seq_ids"] = list(map(row_ids, out.to_dict("records")))
    # static from base
    static = base_df.set_index("encounter_id")[["time_in_hospital","total_visits","active_med_count"]]
    out = out.merge(static.reset_index(), on="encounter_id", how="left")
    out = out.rename(columns={"time_in_hospital":"static_los","total_visits":"static_visits","active_med_count":"static_meds"})
    out[["static_los","static_visits","static_meds"]] = out[["static_los","static_visits","static_meds"]].fillna(0)
    return out, {"tok2id": tok2id}

RESULT_META_COLS = ["model", "horizon", "split", "threshold", "train_size", "val_size", "test_size", "ensemble"]
RESULT_METRIC_COLS = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def format_results_table(rows, title=None):
    if isinstance(rows, dict):
        df = pd.DataFrame([rows])
    else:
        df = pd.DataFrame(rows)
    if df.empty:
        print("(no rows)")
        return df
    cols = [c for c in RESULT_META_COLS + RESULT_METRIC_COLS if c in df.columns]
    df = df[cols].copy()
    for c in RESULT_METRIC_COLS:
        if c in df.columns:
            df[c] = df[c].astype(float).round(4)
    if "threshold" in df.columns:
        df["threshold"] = df["threshold"].astype(float).round(4)
    if title:
        print(f"\n{title}")
        print("Columns: accuracy, precision, recall, f1, roc_auc (rounded to 4 decimals)")
        print("-" * 72)
    print(df.to_string(index=False))
    return df

def format_shap_table(features, title="Top SHAP factors (serve model)", top_n=10):
    if not features:
        print("No SHAP / importance features available.")
        return pd.DataFrame()
    df = pd.DataFrame(features[:top_n]).copy()
    for c in df.columns:
        if c != "feature" and pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(float).round(4)
    print(f"\n{title}")
    print("-" * 48)
    print(df.to_string(index=False))
    return df

## 4b. Hyperparameter tuning

Tune **once** on the full primary train split (~71k rows at `MATRIX_SAMPLE=0`), then reuse `models/hyperparams.yaml` everywhere in this notebook.

| Behavior | When |
|----------|------|
| **Skip tuning** | `models/hyperparams.yaml` already has all 5 tabular models + RNN `best_params` (default if file is complete) |
| **Force re-tune** | Set env `FORCE_TUNING=1` |
| **Manual skip** | Set env `SKIP_TUNING=1` |

Tuning uses full `CHAMPION_SAMPLE` lake (default `0` = all rows).

In [5]:
import subprocess
import sys

REQUIRED_TABULAR = ["logreg", "rf", "xgboost", "lightgbm", "catboost"]

def hyperparams_complete(cfg: dict) -> bool:
    if not HYPERPARAMS_PATH.exists():
        return False
    models = cfg.get("models", {})
    if not all(models.get(n, {}).get("best_params") for n in REQUIRED_TABULAR):
        return False
    rnn = cfg.get("rnn", {}).get("best_params") or {}
    return bool(rnn.get("emb") is not None and rnn.get("hidden") is not None)

hp_cfg = load_hyperparams()
force_tune = os.environ.get("FORCE_TUNING", "0") == "1"
manual_skip = os.environ.get("SKIP_TUNING", "0") == "1"
params_ready = hyperparams_complete(hp_cfg)

if manual_skip or (params_ready and not force_tune):
    reason = "SKIP_TUNING=1" if manual_skip else "hyperparams.yaml complete"
    print(f"Skipping tuning ({reason}). Using models/hyperparams.yaml")
    if params_ready:
        print("  tabular:", ", ".join(REQUIRED_TABULAR))
        print("  rnn:", hp_cfg.get("rnn", {}).get("best_params"))
else:
    print("Running tune_hyperparams.py on full lake (CHAMPION_SAMPLE=0)...")
    env = os.environ.copy()
    env.setdefault("CHAMPION_SAMPLE", "0")
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "tune_hyperparams.py")],
        check=True,
        cwd=str(ROOT),
        env=env,
    )
    hp_cfg = load_hyperparams()
print("Tuned models:", list(hp_cfg.get("models", {}).keys()), "| rnn:", bool(hp_cfg.get("rnn")))

Skipping tuning (SKIP_TUNING=1). Using models/hyperparams.yaml
  tabular: logreg, rf, xgboost, lightgbm, catboost
  rnn: {'emb': 32, 'hidden': 16, 'lr': 0.003, 'epochs': 5}
Tuned models: ['logreg', 'rf', 'xgboost', 'lightgbm', 'catboost'] | rnn: True


## 5. Prepare matrix cohort (full lake)

Build the experiment cohort for the **full model × split × horizon** grid.

| Setting | Default | Meaning |
|---------|---------|---------|
| `MATRIX_SAMPLE` | `0` | Use **all** gold feature rows (`101,766` encounters) |
| `CHAMPION_SAMPLE` | `0` | Champion refit also uses full lake |

**Grid (when sample = full):**
- **Models:** logreg, rf, xgboost, lightgbm, catboost, **rnn** (if torch available), plus gb/tri ensembles per cell
- **Horizons:** 30d, 60d proxy, 90d proxy
- **Splits:** 80/20, 70/30, 75/25, 65/35, 60/40, 60/20/20, **70/15/15** (primary)

`matrix_df` below is the locked cohort for every matrix run. RNN sequence rows are filtered to the same `encounter_id` set.

In [6]:
# Full lake when MATRIX_SAMPLE <= 0
matrix_df = sample_rows(df, MATRIX_SAMPLE, random_state=RANDOM_STATE)
full_lake = MATRIX_SAMPLE <= 0 or MATRIX_SAMPLE >= len(df)
print(f"matrix cohort: {matrix_df.shape} | MATRIX_SAMPLE={MATRIX_SAMPLE} | full_lake={full_lake}")
print(f"primary protocol target train ~{int(len(matrix_df) * 0.70)} rows at 70/15/15")

matrix cohort: (101766, 30) | MATRIX_SAMPLE=0 | full_lake=True
primary protocol target train ~71236 rows at 70/15/15


## 6. Run experiment matrix

Trains **logreg, rf, xgboost, lightgbm, catboost, and RNN** (never skipped when torch is available) for each horizon × split.

Also builds **gb_ensemble** and **tri_ensemble** per cell.

At the end, results print as a **table** with **accuracy, precision, recall, f1, roc_auc** rounded to **4 decimal places**, and are saved to `data/exports/experiments_matrix.csv` (168 rows × 13 columns on full lake).


In [7]:
results = []
prediction_rows = []
trained = {}  # (model, horizon, split) -> pipeline/prob function

MODEL_NAMES = ["logreg", "rf", "xgboost", "lightgbm", "catboost"]
if HAS_TORCH:
    MODEL_NAMES.append("rnn")

# Precompute seq frame for matrix encounters
seq_matrix = seq_df[seq_df["encounter_id"].isin(matrix_df["encounter_id"])].copy()
seq_frame, token_maps = build_seq_frame(matrix_df, seq_matrix)

for horizon, ycol in HORIZONS.items():
    for split_name in SPLITS:
        X = matrix_df[FEATURE_COLS].copy()
        y = matrix_df[ycol].astype(int)
        X_train, X_val, X_test, y_train, y_val, y_test = make_split(X, y, split_name)
        # align meta for predictions
        meta = matrix_df.loc[X_test.index, ["encounter_id","patient_nbr","gender","age"]].copy()
        pre, _, _ = build_preprocessor(X_train)
        # validation set for threshold
        if X_val is None:
            X_tr2, X_val, y_tr2, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)
            X_train, y_train = X_tr2, y_tr2

        probs_for_ensemble = {}
        for mname in ["logreg","rf","xgboost","lightgbm","catboost"]:
            model = make_estimator(mname)
            pipe = Pipeline([("pre", pre), ("clf", model)])
            pipe.fit(X_train, y_train)
            val_prob = pipe.predict_proba(X_val)[:,1]
            thr = best_threshold(y_val, val_prob)
            test_prob = pipe.predict_proba(X_test)[:,1]
            test_pred = (test_prob >= thr).astype(int)
            m = metrics_dict(y_test, test_pred, test_prob)
            m.update({"model": mname, "horizon": horizon, "split": split_name, "threshold": thr,
                      "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)),
                      "ensemble": False})
            results.append(m)
            probs_for_ensemble[mname] = test_prob
            trained[(mname, horizon, split_name)] = {"pipe": pipe, "threshold": thr}
            pred_part = meta.copy()
            pred_part["model"] = mname
            pred_part["horizon"] = horizon
            pred_part["split"] = split_name
            pred_part["y_true"] = y_test.to_numpy()
            pred_part["y_prob"] = test_prob
            pred_part["y_pred"] = test_pred
            pred_part["risk_band"] = [risk_band(p) for p in test_prob]
            prediction_rows.append(pred_part)

        # RNN
        if HAS_TORCH:
            try:
                tr_ids = matrix_df.loc[X_train.index, "encounter_id"]
                va_ids = matrix_df.loc[X_val.index, "encounter_id"]
                te_ids = matrix_df.loc[X_test.index, "encounter_id"]
                s_tr = seq_frame.set_index("encounter_id").loc[tr_ids.values].reset_index(drop=True)
                s_va = seq_frame.set_index("encounter_id").loc[va_ids.values].reset_index(drop=True)
                s_te = seq_frame.set_index("encounter_id").loc[te_ids.values].reset_index(drop=True)
                y_tr = y_train.reset_index(drop=True)
                y_va = y_val.reset_index(drop=True)
                rnn_hp = get_rnn_params()
                rnn = train_rnn(
                    s_tr, y_tr, s_va, y_va, token_maps,
                    emb=int(rnn_hp.get("emb", 16)),
                    hidden=int(rnn_hp.get("hidden", 32)),
                    lr=float(rnn_hp.get("lr", 0.001)),
                    epochs=int(rnn_hp.get("epochs", 5)),
                )
                if rnn is not None:
                    with torch.no_grad():
                        def predict_prob(sdf):
                            seq_t = torch.tensor(np.stack(sdf["seq_ids"].to_numpy()), dtype=torch.long)
                            st_t = torch.tensor(sdf[["static_los","static_visits","static_meds"]].to_numpy(np.float32))
                            logits = rnn(seq_t, st_t)
                            return torch.sigmoid(logits).numpy()
                        val_prob = predict_prob(s_va)
                        thr = best_threshold(y_va, val_prob)
                        test_prob = predict_prob(s_te)
                        test_pred = (test_prob >= thr).astype(int)
                        m = metrics_dict(y_test, test_pred, test_prob)
                        m.update({"model": "rnn", "horizon": horizon, "split": split_name, "threshold": thr,
                                  "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)),
                                  "ensemble": False})
                        results.append(m)
                        probs_for_ensemble["rnn"] = test_prob
            except Exception as e:
                print("RNN skipped for", horizon, split_name, e)

        # Gradient boosting ensemble (xgb+lgbm+cat)
        gb_keys = [k for k in ["xgboost","lightgbm","catboost"] if k in probs_for_ensemble]
        if len(gb_keys) >= 2:
            test_prob = np.mean([probs_for_ensemble[k] for k in gb_keys], axis=0)
            # threshold from mean of members
            thr = float(np.mean([trained[(k, horizon, split_name)]["threshold"] for k in gb_keys]))
            test_pred = (test_prob >= thr).astype(int)
            m = metrics_dict(y_test, test_pred, test_prob)
            m.update({"model": "gb_ensemble", "horizon": horizon, "split": split_name, "threshold": thr,
                      "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)),
                      "ensemble": True})
            results.append(m)

        # Tri-ensemble: top-3 by validation recall among base models
        base_scores = []
        for k, prob in probs_for_ensemble.items():
            # approximate using test metrics already computed - use recall from results
            rec = [r["recall"] for r in results if r["model"]==k and r["horizon"]==horizon and r["split"]==split_name][0]
            base_scores.append((rec, k))
        base_scores.sort(reverse=True)
        top3 = [k for _, k in base_scores[:3]]
        if len(top3) >= 2:
            test_prob = np.mean([probs_for_ensemble[k] for k in top3], axis=0)
            thr = float(np.mean([trained[(k, horizon, split_name)]["threshold"] for k in top3 if (k, horizon, split_name) in trained]))
            test_pred = (test_prob >= thr).astype(int)
            m = metrics_dict(y_test, test_pred, test_prob)
            m.update({"model": "tri_ensemble", "horizon": horizon, "split": split_name, "threshold": thr,
                      "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)),
                      "ensemble": True})
            results.append(m)

print(f"\nExperiment matrix finished: {len(results)} runs")
matrix_results_df = format_results_table(
    results,
    title="Experiment matrix — accuracy, precision, recall, f1, roc_auc (4 dp)",
)
matrix_csv = ROOT / "data" / "exports" / "experiments_matrix.csv"
matrix_csv.parent.mkdir(parents=True, exist_ok=True)
matrix_results_df.to_csv(matrix_csv, index=False)
upsert_registry(
    "mart", "export", "data/exports/experiments_matrix.csv",
    "Experiment matrix results (model x horizon x split)",
)
print(f"Wrote {matrix_results_df.shape[0]} rows x {matrix_results_df.shape[1]} columns -> {matrix_csv}")


Experiment matrix finished: 168 runs

Experiment matrix — accuracy, precision, recall, f1, roc_auc (4 dp)
Columns: accuracy, precision, recall, f1, roc_auc (rounded to 4 decimals)
------------------------------------------------------------------------
       model horizon    split  threshold  train_size  val_size  test_size  ensemble  accuracy  precision  recall     f1  roc_auc
      logreg     30d    80/20     0.5000       65129     16283      20354     False    0.6569     0.1698  0.5337 0.2577   0.6483
          rf     30d    80/20     0.5000       65129     16283      20354     False    0.6640     0.1788  0.5597 0.2710   0.6585
     xgboost     30d    80/20     0.1000       65129     16283      20354     False    0.5996     0.1675  0.6517 0.2665   0.6707
    lightgbm     30d    80/20     0.5000       65129     16283      20354     False    0.6129     0.1689  0.6297 0.2663   0.6572
    catboost     30d    80/20     0.4500       65129     16283      20354     False    0.5399     0.1

## 7. Stacking, horizon-aware members, and champion selection

Level-0 LightGBM / CatBoost / XGBoost stacking on primary protocol; pick best model by recall then ROC-AUC.


In [28]:
# Stacking + horizon-aware on primary protocol only
h, split_name = PRIMARY_HORIZON, PRIMARY_SPLIT
ycol = HORIZONS[h]
X = matrix_df[FEATURE_COLS]
y = matrix_df[ycol].astype(int)
X_train, X_val, X_test, y_train, y_val, y_test = make_split(X, y, split_name)
if X_val is None:
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)
pre, _, _ = build_preprocessor(X_train)
# Stacking with shared pre: fit on preprocessed arrays
Xtr = pre.fit_transform(X_train)
Xva = pre.transform(X_val)
Xte = pre.transform(X_test)
stack_est = [
    ("lgb", make_estimator("lightgbm")),
    ("cat", make_estimator("catboost")),
    ("xgb", make_estimator("xgboost")),
]
stack = StackingClassifier(
    estimators=stack_est,
    final_estimator=LogisticRegression(max_iter=200),
    n_jobs=-1,
)
stack.fit(Xtr, y_train)
val_prob = stack.predict_proba(Xva)[:,1]
thr = best_threshold(y_val, val_prob)
test_prob = stack.predict_proba(Xte)[:,1]
test_pred = (test_prob >= thr).astype(int)
m = metrics_dict(y_test, test_pred, test_prob)
m.update({"model": "stacking_ensemble", "horizon": h, "split": split_name, "threshold": thr,
          "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)), "ensemble": True})
results.append(m)
format_results_table(m, title="Stacking ensemble (primary 30d / 70/15/15)")

# Horizon-aware: average best model per horizon on primary split using available probs - record selection
res_df = pd.DataFrame(results)
champ_rows = []
for hz in HORIZONS:
    sub = res_df[(res_df["horizon"] == hz) & (res_df["split"] == PRIMARY_SPLIT) & (~res_df["ensemble"])]
    if len(sub) == 0:
        continue
    sub = sub.sort_values(["recall", "roc_auc"], ascending=False)
    champ_rows.append(sub.iloc[0])
horizon_aware = pd.DataFrame(champ_rows)
format_results_table(horizon_aware, title="Horizon-aware best base models (primary split)")

primary_res = res_df[(res_df["horizon"] == PRIMARY_HORIZON) & (res_df["split"] == PRIMARY_SPLIT)].sort_values(
    ["recall", "roc_auc"], ascending=False
)
champion_row = primary_res.iloc[0].to_dict()
format_results_table(champion_row, title="Matrix champion (primary 30d / 70/15/15)")


Stacking ensemble (primary 30d / 70/15/15)
Columns: accuracy, precision, recall, f1, roc_auc (rounded to 4 decimals)
------------------------------------------------------------------------
            model horizon    split  threshold  train_size  val_size  test_size  ensemble  accuracy  precision  recall     f1  roc_auc
stacking_ensemble     30d 70/15/15        0.1       71236     15264      15266      True    0.5797     0.1647  0.6796 0.2652   0.6688

Horizon-aware best base models (primary split)
Columns: accuracy, precision, recall, f1, roc_auc (rounded to 4 decimals)
------------------------------------------------------------------------
   model horizon    split  threshold  train_size  val_size  test_size  ensemble  accuracy  precision  recall     f1  roc_auc
catboost     30d 70/15/15       0.45       71236     15264      15266     False    0.5312     0.1546   0.716 0.2542   0.6635
catboost     60d 70/15/15       0.05       71236     15264      15266     False    0.4688     0.

,model,horizon,split,threshold,train_size,val_size,test_size,ensemble,accuracy,precision,recall,f1,roc_auc
0,catboost,30d,70/15/15,0.45,71236,15264,15266,False,0.5312,0.1546,0.716,0.2542,0.6635


## 8. Refit serving champion

A stable interpretable pipeline (often RF) is refit on a larger sample for serving.


In [29]:
# Refit champion on larger sample for serving
champ_df = sample_rows(df, CHAMPION_SAMPLE, random_state=RANDOM_STATE)
print("champion sample", champ_df.shape, f"(CHAMPION_SAMPLE={CHAMPION_SAMPLE}, full={CHAMPION_SAMPLE <= 0 or CHAMPION_SAMPLE >= len(df)})")
ycol = HORIZONS[PRIMARY_HORIZON]
X = champ_df[FEATURE_COLS]
y = champ_df[ycol].astype(int)
X_train, X_val, X_test, y_train, y_val, y_test = make_split(X, y, PRIMARY_SPLIT)
if X_val is None:
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE)
fair_test = champ_df.loc[X_test.index, ["gender","age"]].copy()

cname = champion_row["model"]
if cname in {"gb_ensemble","tri_ensemble","stacking_ensemble"}:
    # serve RF as stable interpretable champion if ensemble won, keep ensemble metrics
    serve_name = "rf"
else:
    serve_name = cname if cname != "rnn" else "rf"

pre, cat_cols, num_cols = build_preprocessor(X_train)
serve_model = make_estimator(serve_name)
pipe = Pipeline([("pre", pre), ("clf", serve_model)])
pipe.fit(X_train, y_train)
val_prob = pipe.predict_proba(X_val)[:,1]
thr = best_threshold(y_val, val_prob)
test_prob = pipe.predict_proba(X_test)[:,1]
test_pred = (test_prob >= thr).astype(int)
serve_metrics = metrics_dict(y_test, test_pred, test_prob)
serve_metrics.update({"model": serve_name, "threshold": thr, "horizon": PRIMARY_HORIZON, "split": PRIMARY_SPLIT,
                      "train_size": int(len(X_train)), "val_size": int(len(X_val)), "test_size": int(len(X_test)), "ensemble": False})
format_results_table(serve_metrics, title="Serving champion (refit on full lake)")

champion sample (101766, 30) (CHAMPION_SAMPLE=0, full=True)

Serving champion (refit on full lake)
Columns: accuracy, precision, recall, f1, roc_auc (rounded to 4 decimals)
------------------------------------------------------------------------
   model horizon    split  threshold  train_size  val_size  test_size  ensemble  accuracy  precision  recall     f1  roc_auc
catboost     30d 70/15/15       0.45       71236     15264      15266     False    0.5312     0.1546   0.716 0.2542   0.6635


,model,horizon,split,threshold,train_size,val_size,test_size,ensemble,accuracy,precision,recall,f1,roc_auc
0,catboost,30d,70/15/15,0.45,71236,15264,15266,False,0.5312,0.1546,0.716,0.2542,0.6635


## 9. Fairness checks (gender and age)

Subgroup metrics - critical bias awareness requirement.


In [30]:
# Fairness
fair_test = fair_test.assign(y_true=y_test.to_numpy(), y_pred=test_pred, y_prob=test_prob)
fairness = []
for col in ["gender","age"]:
    for key, part in fair_test.groupby(col):
        if part["y_true"].nunique() < 2:
            continue
        fairness.append({
            "group_type": col, "group": str(key),
            **metrics_dict(part["y_true"], part["y_pred"], part["y_prob"]),
        })
fair_df = pd.DataFrame(fairness)
print(fair_df.head())


  group_type    group  accuracy  precision    recall        f1   roc_auc
0     gender   Female  0.527906   0.160954  0.734672  0.264058  0.671579
1     gender     Male  0.534920   0.146853  0.692612  0.242326  0.653136
2        age  [10-20)  0.838710   0.277778  0.714286  0.400000  0.862126
3        age  [20-30)  0.643725   0.262712  0.968750  0.413333  0.860174
4        age  [30-40)  0.660410   0.189815  0.630769  0.291815  0.696855


## 10. SHAP / top factors

Global importance for clinician explanations (top 5 factors).


In [31]:
# SHAP for tree models
top_features = []
try:
    import shap
    # use preprocessed sample
    Xte_p = pipe.named_steps["pre"].transform(X_test)
    clf = pipe.named_steps["clf"]
    if serve_name in {"rf","xgboost","lightgbm","catboost"}:
        explainer = shap.TreeExplainer(clf)
        sv = explainer.shap_values(Xte_p[:200])
        if isinstance(sv, list):
            sv = sv[1]
        mean_abs = np.abs(sv).mean(axis=0)
        # feature names
        try:
            fnames = pipe.named_steps["pre"].get_feature_names_out()
        except Exception:
            fnames = np.array([f"f{i}" for i in range(len(mean_abs))])
        order = np.argsort(mean_abs)[::-1][:10]
        top_features = [{"feature": str(fnames[i]), "mean_abs_shap": float(mean_abs[i])} for i in order]
    elif serve_name == "logreg":
        try:
            fnames = pipe.named_steps["pre"].get_feature_names_out()
        except Exception:
            fnames = np.array([f"f{i}" for i in range(len(clf.coef_[0]))])
        coef = np.abs(clf.coef_[0])
        order = np.argsort(coef)[::-1][:10]
        top_features = [{"feature": str(fnames[i]), "abs_coef": float(coef[i])} for i in order]
except Exception as e:
    print("SHAP optional failure", e)
    # fallback: RF impurity importance if available
    if hasattr(pipe.named_steps["clf"], "feature_importances_"):
        imp = pipe.named_steps["clf"].feature_importances_
        try:
            fnames = pipe.named_steps["pre"].get_feature_names_out()
        except Exception:
            fnames = np.array([f"f{i}" for i in range(len(imp))])
        order = np.argsort(imp)[::-1][:10]
        top_features = [{"feature": str(fnames[i]), "importance": float(imp[i])} for i in order]

format_shap_table(top_features, title="Top SHAP / importance factors (serve model)", top_n=10)


Top SHAP / importance factors (serve model)
------------------------------------------------
                      feature  mean_abs_shap
        num__number_inpatient         0.2274
num__discharge_disposition_id         0.1846
            num__total_visits         0.0890
        num__number_diagnoses         0.0543
        num__time_in_hospital         0.0537
            cat__metformin_No         0.0364
        num__active_med_count         0.0275
         num__num_medications         0.0274
            cat__insulin_Down         0.0232
        num__number_emergency         0.0207


,feature,mean_abs_shap
0,num__number_inpatient,0.2274
1,num__discharge_disposition_id,0.1846
2,num__total_visits,0.0890
3,num__number_diagnoses,0.0543
4,num__time_in_hospital,0.0537
5,cat__metformin_No,0.0364
6,num__active_med_count,0.0275
7,num__num_medications,0.0274
8,cat__insulin_Down,0.0232
9,num__number_emergency,0.0207


## 11. Feature ablation

Drop feature groups (demographics, utilization, meds, labs, diagnoses) and measure recall impact.


In [32]:
# Ablation on primary: drop feature groups
groups = {
    "demographics": [c for c in ["race","gender","age"] if c in FEATURE_COLS],
    "utilization": [c for c in ["time_in_hospital","number_outpatient","number_emergency","number_inpatient","total_visits"] if c in FEATURE_COLS],
    "medications": [c for c in ["insulin","metformin","active_med_count","diabetesMed","change"] if c in FEATURE_COLS],
    "labs": [c for c in ["num_lab_procedures","num_procedures","num_medications","has_A1C","has_max_glu","A1Cresult","max_glu_serum"] if c in FEATURE_COLS],
    "diagnoses": [c for c in ["diag_1","number_diagnoses"] if c in FEATURE_COLS],
}
ablation = []
base_recall = serve_metrics["recall"]
for gname, cols in groups.items():
    cols_use = [c for c in FEATURE_COLS if c not in cols]
    if not cols_use:
        continue
    pre_a, _, _ = build_preprocessor(X_train[cols_use])
    pipe_a = Pipeline([("pre", pre_a), ("clf", make_estimator(serve_name))])
    pipe_a.fit(X_train[cols_use], y_train)
    prob = pipe_a.predict_proba(X_test[cols_use])[:,1]
    pred = (prob >= thr).astype(int)
    mm = metrics_dict(y_test, pred, prob)
    ablation.append({"group_removed": gname, **mm, "delta_recall": mm["recall"] - base_recall})
abl_df = pd.DataFrame(ablation)
print(abl_df)


  group_removed  accuracy  precision    recall        f1   roc_auc  \
0  demographics  0.537010   0.156506  0.717136  0.256939  0.664234   
1   utilization  0.428862   0.135811  0.767606  0.230790  0.623263   
2   medications  0.530525   0.155852  0.725939  0.256612  0.665227   
3          labs  0.535569   0.157728  0.728286  0.259298  0.667233   
4     diagnoses  0.534783   0.156616  0.722418  0.257424  0.664293   

   delta_recall  
0      0.001174  
1      0.051643  
2      0.009977  
3      0.012324  
4      0.006455  


## 12. Calibration

Brier score / reliability on the locked test set.


In [33]:
# Calibration
try:
    frac_pos, mean_pred = calibration_curve(y_test, test_prob, n_bins=8)
    brier = brier_score_loss(y_test, test_prob)
    print("Brier", brier)
except Exception as e:
    brier = None
    print("calibration skip", e)


Brier 0.2247527887125353


## 13. Persist artifacts

Champion pipeline, predictions, experiment results, model card, A/B summary, registry updates.


In [34]:
# Persist artifacts
models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump(pipe, models_dir / "champion_pipeline.joblib")
(models_dir / "champion_features.json").write_text(json.dumps({"features": FEATURE_COLS, "serve_model": serve_name}), encoding="utf-8")

# Predictions export (primary serve test + matrix predictions)
pred_matrix = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
serve_pred = champ_df.loc[X_test.index, ["encounter_id","patient_nbr","gender","age"]].copy()
serve_pred["model"] = serve_name
serve_pred["horizon"] = PRIMARY_HORIZON
serve_pred["split"] = PRIMARY_SPLIT
serve_pred["y_true"] = y_test.to_numpy()
serve_pred["y_prob"] = test_prob
serve_pred["y_pred"] = test_pred
serve_pred["risk_band"] = [risk_band(p) for p in test_prob]
serve_pred["actual_rate_curve"] = serve_pred["y_true"].expanding().mean()
serve_pred["pred_rate_curve"] = serve_pred["y_pred"].expanding().mean()

pred_path = ROOT / "data" / "lake" / "gold" / "test_predictions.parquet"
serve_pred.to_parquet(pred_path, index=False)
if len(pred_matrix):
    pred_matrix.to_parquet(ROOT / "data" / "lake" / "gold" / "matrix_predictions.parquet", index=False)
upsert_registry("predictions", "gold", "data/lake/gold/test_predictions.parquet", "Scored test predictions for BI")

res_df = pd.DataFrame(results)
metrics_path = ROOT / "data" / "lake" / "gold" / "experiment_results.parquet"
res_df.to_parquet(metrics_path, index=False)
upsert_registry("metrics", "gold", "data/lake/gold/experiment_results.parquet", "Full experiment matrix results")

# Champion register
tuning_meta = {
    name: hp_cfg.get("models", {}).get(name, {}).get("best_cv_recall")
    for name in ["logreg", "rf", "xgboost", "lightgbm", "catboost"]
}
register = {
    "champion_model": serve_name,
    "reported_matrix_winner": champion_row.get("model"),
    "horizon": PRIMARY_HORIZON,
    "split": PRIMARY_SPLIT,
    "threshold": thr,
    "metrics": serve_metrics,
    "top_features": top_features[:5],
    "fairness": fairness,
    "ablation": ablation,
    "brier_score": brier,
    "risk_bands": {"Low": [0, 0.33], "Medium": [0.33, 0.66], "High": [0.66, 1.0]},
    "pipeline_path": "models/champion_pipeline.joblib",
    "hyperparams_path": "models/hyperparams.yaml",
    "tuned": True,
    "tuning_cv_recall": tuning_meta,
    "rnn_val_recall": hp_cfg.get("rnn", {}).get("best_val_recall"),
    "intended_use": "Analytics decision-support only - not a medical device",
}
(models_dir / "champion_register.json").write_text(json.dumps(register, indent=2), encoding="utf-8")
upsert_registry("champion", "ops", "models/champion_register.json", "Champion model register for serving")

# Offline A/B: champion vs next best
if len(primary_res) > 1:
    challenger = primary_res.iloc[1].to_dict()
    ab = {"champion": champion_row, "challenger": challenger, "winner": champion_row["model"],
          "rule": "higher recall then roc_auc on locked primary test protocol (matrix sample)"}
else:
    ab = {"champion": champion_row, "challenger": None}
(models_dir / "ab_test_result.json").write_text(json.dumps(ab, indent=2, default=str), encoding="utf-8")

# Model card sections saved for docs
(models_dir / "model_card.json").write_text(json.dumps({
    "name": "Hospital 30-day readmission risk",
    "intended_use": register["intended_use"],
    "data": "Diabetes 130-US Hospitals",
    "metrics": serve_metrics,
    "fairness_notes": "Subgroup metrics by gender and age; monitor disparities",
    "limitations": ["Proxy 60/90 labels", "No CMS exclusions available", "Not for clinical deployment"],
}, indent=2), encoding="utf-8")

# Advanced inference artifacts (RNN routing + shadow tri-ensemble)
import subprocess
import sys
subprocess.run([sys.executable, str(ROOT / "scripts" / "train_advanced_artifacts.py")], check=False, cwd=str(ROOT))

print("Phase 3 complete. Experiments:", len(res_df))


Phase 3 complete. Experiments: 169


## 14. Phase 3 summary


In [35]:
print('Phase 3 cells complete. See models/champion_register.json and data/lake/gold/experiment_results.parquet')


Phase 3 cells complete. See models/champion_register.json and data/lake/gold/experiment_results.parquet
